#### **Problem Statement :**

- For complex reasoning tasks where simpler and traditional approaches may not be sufficient, tree-of-thoughts prompting technique has been recently proposed in literature, which encourages trial and error and backtracking when necessary [1][2].

- Reference [2] has shown the tree of thoughts approach can significantly improve the ability of an llm to solve Sudoku Puzzle.

- LLM token generation is typically based on preceding sequence of tokens without backward editing. However when a human solves a problem, one attempts to backtrack to the previous step to make sure the derivations are correct [2].

- This approach of backtracking possibly negates the LLM from reaching an inconclusive response.

- Additionally this approach establishes credibility just as humans ensure correctness by carrying out tests at everystep of solving a problem.

- Tree of thoughts agents implemented by LangChain is shown.

- Each thought is evaluated by a ToT Checker and is categorized to be valid, invalid or intermediate [5].

#### **References**

1.  **Yao, Shunyu, et al. "Tree of thoughts: Deliberate problem solving with large language models." arXiv preprint arXiv:2305.10601 (2023)**

2. **Long, Jieyi. "Large Language Model Guided Tree-of-Thought." arXiv preprint arXiv:2305.08291 (2023).**

3. https://python.langchain.com/cookbook

4. https://www.promptingguide.ai/techniques/tot

5. **Chase, Harrison. "LangChain, October 2022." URL https://github.com/hwchase17/langchain 3 (2022).** Acknowledged.

#### **Installing dependencies**

In [ ]:
# Installing the required libraries
# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q langchain==0.3.27 langchain-community==0.3.27 langchain-core==0.3.72 langchain-text-splitters==0.3.9 langchain-openai==0.3.28 openai==1.99.9 tiktoken==0.14.0 langchain-experimental==0.3.4
# !pip install --upgrade openai -q   # (replaced by the pinned install above)
# !pip install langchain -q   # (replaced by the pinned install above)
# !pip install langchain_experimental -q   # (replaced by the pinned install above)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.
tensorflow-probability 0.22.0 requires typing-extensions<4.6.0, but you have typing-extensions 4.9.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.6/803.6 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.9/235.9 kB 21.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━

#### **Importing required packages**

In [ ]:
# Importing the required libraries

from google.colab import userdata
from openai import OpenAI
import openai

#### **OpenAI API KEY**

In [ ]:
# Providing the OPENAI API KEY

OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')
openai.api_key = OPENAI_API_KEY


client = OpenAI(
    api_key = OPENAI_API_KEY
)

In [ ]:
# Client.chat.completions.create() function where the model is specified

chat_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system","content":"You are a helpful assistant"},
        {"role": "user", "content": "What is artificial intelligence?"},
        {"role": "system", "content": "Artifical intelligence is the simulation of human intelligence processes by computer systems"},
        {"role": "user", "content": "What are two applications of artifical intelligence?"}
]
)

In [ ]:
# Response from the model

chat_completion.choices[0].message.content

"1. Speech Recognition: Some intelligent systems are capable of understanding and responding to human speech, like virtual assistants such as Amazon's Alexa, Google Home, or Apple's Siri.\n\n2. Autonomous Vehicles: AI is used in self-driving cars to interpret and respond to their surroundings in real time. It allows these vehicles to make decisions, like when to slow down, speed up, or overtake other vehicles, based on ongoing sensory input."

In [ ]:
# Langchain implemented guided tree-of-thoughts approach

from langchain_openai import ChatOpenAI as OpenAI  # completion model retired; chat model used instead
llm = OpenAI(openai_api_key=OPENAI_API_KEY, max_tokens=512, temperature=1, model = "gpt-4o-mini")

#######

sudoko_puzzle = "3,*,*,2|1,*,3,*|*,1,*,3|4,*,*,1"
sudoku_solution = "3,4,1,2|1,2,3,4|2,1,4,3|4,3,2,1"

problem_description = f"""
{sudoko_puzzle}

- This is a 4x4 Sudoku puzzle.
- The * represents a cell to be filled.
- The | character separates rows.
- At each step, replace one or more * with digits 1-4.
- There must be no duplicate digits in any row, column or 2x2 subgrid.
- Keep the known digits from previous valid thoughts in place.
- Each thought can be a partial or the final solution.
""".strip()
print(problem_description)

#######
# The following code implements a rule based checker for 4x4 sudoku puzzle
#######

from typing import Tuple
from langchain_experimental.tot.checker import ToTChecker
from langchain_experimental.tot.thought import ThoughtValidity
import re

class MyChecker(ToTChecker):
  def evaluate(
      self, problem_description: str, thoughts: Tuple[str, ...] = ()
      ) -> ThoughtValidity:
      last_thought = thoughts[-1]
      clean_solution = last_thought.replace(" ", "").replace('"', "")
      regex_solution = clean_solution.replace("*", ".").replace("|", "\\|")
      if sudoku_solution in clean_solution:
        return ThoughtValidity.VALID_FINAL
      elif re.search(regex_solution, sudoku_solution):
        return ThoughtValidity.VALID_INTERMEDIATE
      else:
        return ThoughtValidity.INVALID

#######
# testing the MyChecker class above
#######

checker = MyChecker()
assert checker.evaluate("", ("3,*,*,2|1,*,3,*|*,1,*,3|4,*,*,1",)) == ThoughtValidity.VALID_INTERMEDIATE
assert checker.evaluate("", ("3,4,1,2|1,2,3,4|2,1,4,3|4,3,2,1",)) == ThoughtValidity.VALID_FINAL
assert checker.evaluate("", ("3,4,1,2|1,2,3,4|2,1,4,3|4,*,*,1",)) == ThoughtValidity.VALID_INTERMEDIATE
assert checker.evaluate("", ("3,4,1,2|1,2,4|2,1,4,3|4,*,3,1",)) == ThoughtValidity.INVALID

#######
#Initialize and run ToT Chain
# with maximum number of interactions k set to 30 and
# the maximum number of child thoughts c set to 8
#######

from langchain_experimental.tot.base import ToTChain

tot_chain = ToTChain(llm=llm, checker=MyChecker(), k=30, c=5, verbose=True, verbose_llm=False)
tot_chain.run(problem_description=problem_description)


#######

3,*,*,2|1,*,3,*|*,1,*,3|4,*,*,1

- This is a 4x4 Sudoku puzzle.
- The * represents a cell to be filled.
- The | character separates rows.
- At each step, replace one or more * with digits 1-4.
- There must be no duplicate digits in any row, column or 2x2 subgrid.
- Keep the known digits from previous valid thoughts in place.
- Each thought can be a partial or the final solution.


> Entering new ToTChain chain...
Starting the ToT solve procedure.


/usr/local/lib/python3.10/dist-packages/langchain/chains/llm.py:316: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


Thought: 1,*,*,2|1,*,3,*|*,1,*,3|4,*,*,1
Thought: 3,*,*,2|1,*,3,*|1,1,*,3|4,*,*,1
Thought: 3,*,4,2|1,*,3,*|1,1,*,3|4,*,*,1
Thought: 3,*,4,2|1,*,3,*|1,1,1,3|4,*,*,1
Thought: 3,*,4,2|1,*,3,*|1,1,1,3|4,*,2,1


/usr/local/lib/python3.10/dist-packages/langchain/chains/llm.py:316: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


Thought: 2,*,*,2|1,*,3,*|*,1,*,3|4,*,*,1
Thought: 3,2,*,2|1,*,3,*|*,1,*,3|4,*,*,1
Thought: 3,2,*,2|1,*,3,*|*,1,*,3|4,*,*,1
Thought: 3,2,2,2|1,*,3,*|*,1,*,3|4,*,*,1
Thought: 3,2,3,2|1,*,3,*|*,1,*,3|4,*,*,1


/usr/local/lib/python3.10/dist-packages/langchain/chains/llm.py:316: UserWarning: The predict_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


Thought: 3,4,1,2|1,2,3,4|2,1,4,3|4,3,2,1

> Finished chain.


'3,4,1,2|1,2,3,4|2,1,4,3|4,3,2,1'